# Data Skew in Spark

An in-depth PySpark DataFrame lab with multi-million-row customer, order, and order-item data staged in HDFS. We expose skew with AQE disabled, apply explicit solutions, then enable AQE and inspect runtime skew-join handling.

> Expected: Spark 3.5+ and HDFS at `hdfs://localhost:9000`. Run cells in order.

## Learning goals

- Generate deterministic 99/1 geographic skew without driver-side row creation.
- Distinguish source, shuffle-key, compute, and output skew.
- Diagnose skew using key counts, partition counts, plans, and Spark UI metrics.
- Compare country aggregation, country join, and customer-key join behavior.
- Solve skew with broadcasting, deterministic salting, and meaningful `(country, state)` hierarchy.
- Understand exactly when AQE detects skew and what AQE cannot repair.

# 1. What skew means

Hash partitioning sends identical keys to the same shuffle partition. If 99% of rows have `country='India'`, a shuffle by country can resemble `[99%, <1%, <1%, <1%]`. A stage completes only when its slowest task finishes, so one reducer can keep the entire stage waiting. More executors or more partitions do not split one identical key by themselves.

Skew can mean uneven source files, uneven shuffle keys, uneven per-row computation, or uneven output files. This lab focuses on shuffle-key skew. Key frequency is a warning—not proof of slow execution—because partial aggregation can drastically reduce rows before a shuffle.

# 2. Spark initialization: AQE off

AQE and automatic broadcasting begin disabled. That makes the initial shuffle behavior visible. Later, broadcasting is shown as the preferred fix for a tiny lookup, and AQE is enabled explicitly.

In [ ]:
from time import perf_counter

from pyspark.sql import SparkSession, functions as F

HDFS = "hdfs://localhost:9000"
STAGE_ROOT = f"{HDFS}/tmp/dataeng/skew_lab"
CUSTOMER_ROWS = 2_000_000
ORDER_ROWS = 8_000_000
ITEMS_PER_ORDER = 3  # Produces 24 million order-item rows.
SOURCE_PARTITIONS = 48
SHUFFLE_PARTITIONS = 24
INDIA_PERCENT = 99  # Use 90 for milder skew.
SALT_BUCKETS = 16

spark = (
    SparkSession.builder
    .appName("Spark-Skew-Lab")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", HDFS)
    .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print(
    f"customers={CUSTOMER_ROWS:,}; "
    f"orders={ORDER_ROWS:,}; "
    f"items={ORDER_ROWS * ITEMS_PER_ORDER:,}"
)

## Scale guidance

The defaults satisfy the multi-million-row requirement. For a laptop validation run, use 200,000 customers and 800,000 orders while retaining the ratios. On a cluster, increase volume until the hot shuffle partition produces visible stragglers or spill. Row width, compression, cores, and memory matter as much as row count.

# 3. Generate customers with country and state skew

For every 100 IDs, 99 become India. The remaining 1% rotates across Sri Lanka, Nepal, and Bhutan, so each receives less than 1%. India is subdivided across 36 synthetic state keys. The distribution is for teaching, not demographic modeling.

In [ ]:
position = F.pmod(F.col("id"), F.lit(100))
minority_index = F.pmod(F.col("id"), F.lit(3))

country_expression = (
    F.when(position < INDIA_PERCENT, "India")
    .when(minority_index == 0, "Sri Lanka")
    .when(minority_index == 1, "Nepal")
    .otherwise("Bhutan")
)

state_expression = (
    F.when(
        F.col("country") == "India",
        F.format_string("IN_%02d", F.pmod("id", F.lit(36))),
    )
    .when(
        F.col("country") == "Sri Lanka",
        F.format_string("LK_%02d", F.pmod("id", F.lit(9))),
    )
    .when(
        F.col("country") == "Nepal",
        F.format_string("NP_%02d", F.pmod("id", F.lit(7))),
    )
    .otherwise(
        F.format_string("BT_%02d", F.pmod("id", F.lit(5)))
    )
)

customers_generated = (
    spark.range(CUSTOMER_ROWS, numPartitions=SOURCE_PARTITIONS)
    .withColumn("country", country_expression)
    .select(
        F.col("id").alias("customer_id"),
        F.col("country"),
        state_expression.alias("state"),
        F.when(F.pmod("id", F.lit(1000)) < 10, "VIP")
        .otherwise("STANDARD")
        .alias("segment"),
    )
)

(
    customers_generated
    .groupBy("country")
    .count()
    .orderBy(F.desc("count"))
    .show()
)

# 4. Generate orders and order items

Orders reference customers, so geographic skew appears after enrichment. Twenty percent deliberately use `customer_id=0`, creating a separate hot-customer join key for the AQE lesson. Items store `product_id`, `quantity`, and unit `price`; amount is computed rather than stored.

In [ ]:
orders_generated = (
    spark.range(ORDER_ROWS, numPartitions=SOURCE_PARTITIONS)
    .select(
        F.col("id").alias("order_id"),
        F.when(F.pmod("id", F.lit(5)) == 0, F.lit(0))
        .otherwise(
            F.pmod(F.col("id") * 7919, F.lit(CUSTOMER_ROWS))
        )
        .alias("customer_id"),
        F.date_add(
            F.lit("2025-01-01").cast("date"),
            F.pmod("id", F.lit(365)).cast("int"),
        ).alias("order_date"),
        F.element_at(
            F.array(F.lit("APP"), F.lit("WEB"), F.lit("STORE")),
            (F.pmod("id", F.lit(3)) + 1).cast("int"),
        ).alias("channel"),
    )
)

order_items_generated = (
    orders_generated
    .select("order_id")
    .withColumn(
        "line_number",
        F.explode(F.sequence(F.lit(1), F.lit(ITEMS_PER_ORDER))),
    )
    .select(
        F.col("order_id"),
        F.col("line_number"),
        F.pmod(
            F.col("order_id") * 31 + F.col("line_number"),
            F.lit(50_000),
        ).alias("product_id"),
        (
            F.pmod(
                F.col("order_id") + F.col("line_number"),
                F.lit(5),
            )
            + 1
        ).cast("int").alias("quantity"),
        F.round(
            F.lit(5.0)
            + F.pmod(
                F.col("order_id") * 17 + F.col("line_number"),
                F.lit(20_000),
            ) / 100,
            2,
        ).alias("price"),
    )
)

(
    orders_generated
    .groupBy("customer_id")
    .count()
    .orderBy(F.desc("count"))
    .show(5)
)

# 5. Stage clean Parquet data in HDFS

Staging separates generation from measurement. Customers are intentionally not partitioned by country because that would mix source-layout skew with shuffle skew. Orders use year/month for realistic date pruning. The overwrite is limited to the dedicated lab root.

In [ ]:
assert STAGE_ROOT == "hdfs://localhost:9000/tmp/dataeng/skew_lab"

CUSTOMERS_PATH = f"{STAGE_ROOT}/customers"
ORDERS_PATH = f"{STAGE_ROOT}/orders"
ITEMS_PATH = f"{STAGE_ROOT}/order_items"

(
    customers_generated.write
    .mode("overwrite")
    .parquet(CUSTOMERS_PATH)
)

(
    orders_generated
    .withColumn("order_year", F.year("order_date"))
    .withColumn("order_month", F.month("order_date"))
    .write
    .mode("overwrite")
    .partitionBy("order_year", "order_month")
    .parquet(ORDERS_PATH)
)

(
    order_items_generated.write
    .mode("overwrite")
    .parquet(ITEMS_PATH)
)

customers = spark.read.parquet(CUSTOMERS_PATH)
orders = spark.read.parquet(ORDERS_PATH)
order_items = spark.read.parquet(ITEMS_PATH)

print(CUSTOMERS_PATH)
print(ORDERS_PATH)
print(ITEMS_PATH)

# 6. Compute order amount and build the fact

Aggregate items before joining so 24 million line items become 8 million order totals. Spark can partially aggregate before shuffling. Joining customers by `customer_id` is correct. Geographic skew alone does not skew this join because hashing uses `customer_id`; the deliberately hot customer can skew it.

The enriched fact is written once and read back. This materializes the expensive item aggregation and joins, so later experiments measure the country operation instead of repeatedly rebuilding the fact lineage.

In [ ]:
order_totals = (
    order_items
    .select(
        F.col("order_id"),
        (F.col("quantity") * F.col("price")).alias("line_amount"),
    )
    .groupBy("order_id")
    .agg(F.sum("line_amount").alias("order_amount"))
)

order_fact_build = (
    orders
    .select("order_id", "customer_id", "order_date", "channel")
    .join(order_totals, "order_id")
    .join(
        customers.select("customer_id", "country", "state"),
        "customer_id",
    )
    .select(
        "order_id",
        "customer_id",
        "country",
        "state",
        "order_amount",
        "order_date",
        "channel",
    )
)

order_fact_build.explain(mode="formatted")

ORDER_FACT_PATH = f"{STAGE_ROOT}/order_fact"

(
    order_fact_build.write
    .mode("overwrite")
    .parquet(ORDER_FACT_PATH)
)

order_fact = spark.read.parquet(ORDER_FACT_PATH)
print("Staged enriched fact:", ORDER_FACT_PATH)

# 7. Diagnose before tuning

Frequency counts expose semantic skew. The partition profile explicitly shuffles by a key and reports physical rows per non-empty partition. In the Spark UI compare maximum with median task duration and shuffle-read bytes; also inspect spill, peak memory, and GC time. A long task tail is stronger evidence than total runtime alone.

In [ ]:
def key_profile(dataframe, *keys, top=20):
    (
        dataframe
        .groupBy(*keys)
        .count()
        .orderBy(F.desc("count"))
        .show(top, truncate=False)
    )


def shuffled_partition_profile(dataframe, *keys):
    return (
        dataframe
        .repartition(SHUFFLE_PARTITIONS, *keys)
        .withColumn("partition_id", F.spark_partition_id())
        .groupBy("partition_id")
        .count()
        .orderBy(F.desc("count"))
    )


key_profile(customers, "country")

country_partition_profile = shuffled_partition_profile(
    customers.select("country"),
    "country",
)
country_partition_profile.show(SHUFFLE_PARTITIONS, truncate=False)

# 8. Country-wise revenue: skewed key, but partial aggregation helps

`groupBy('country')` ultimately sends each country to one final key. However, `sum` and `count` are associative: Spark normally emits a partial `HashAggregate` before the exchange, so each source partition sends only a few partial rows. Country frequency can therefore look terrible while the final shuffle remains small.

Aggregation skew is more damaging when state cannot shrink locally—for example `collect_list`, exact high-cardinality distinct processing, grouped Pandas UDFs, or expensive arbitrary per-group logic. AQE's skew-join rule does not generally split one final aggregation key.

In [ ]:
country_revenue = (
    order_fact
    .groupBy("country")
    .agg(
        F.sum("order_amount").alias("revenue"),
        F.count("*").alias("orders"),
    )
)

country_revenue.explain(mode="formatted")

(
    country_revenue
    .orderBy(F.desc("revenue"))
    .show(truncate=False)
)

# 9. Two-stage aggregation with deterministic salt

Create `(country, salt)` using a stable high-cardinality ID, aggregate partial results, then remove salt with a second aggregation. India is distributed across `SALT_BUCKETS` groups while the final answer stays exact. This adds an aggregation and shuffle; it may be slower than native partial `sum`, so use it when measurement shows large retained per-key work.

In [ ]:
salted_partial = (
    order_fact
    .withColumn(
        "salt",
        F.pmod(F.xxhash64("order_id"), F.lit(SALT_BUCKETS)),
    )
    .groupBy("country", "salt")
    .agg(
        F.sum("order_amount").alias("partial_revenue"),
        F.count("*").alias("partial_orders"),
    )
)

salted_country = (
    salted_partial
    .groupBy("country")
    .agg(
        F.sum("partial_revenue").alias("revenue"),
        F.sum("partial_orders").alias("orders"),
    )
)

salted_country.explain(mode="formatted")

(
    salted_country
    .orderBy(F.desc("revenue"))
    .show(truncate=False)
)

# 10. Meaningful subdivision: country + state

If state-level subtotals are valid, `(country, state)` naturally spreads India across 36 groups; roll those totals up to country afterward. Unlike arbitrary salt, the intermediate result has business meaning. But a composite join key is correct only when both relations represent the same state-level entity. Adding state to a country-only join can change matches and silently change the answer.

In [ ]:
state_partial = (
    order_fact
    .groupBy("country", "state")
    .agg(
        F.sum("order_amount").alias("state_revenue"),
        F.count("*").alias("state_orders"),
    )
)

state_country = (
    state_partial
    .groupBy("country")
    .agg(
        F.sum("state_revenue").alias("revenue"),
        F.sum("state_orders").alias("orders"),
    )
)

key_profile(order_fact, "country", "state", top=45)

(
    state_country
    .orderBy(F.desc("revenue"))
    .show(truncate=False)
)

# 11. Country-key shuffle join

A four-row country rule lookup is joined to the fact. Because broadcast is temporarily disabled, Spark must shuffle by country and roughly 99% of fact rows reach India's reducer. Never join two fact-like tables only by country: that can produce `left_count(country) × right_count(country)` rows, a modeling error rather than merely a tuning problem.

In [ ]:
country_rules = spark.createDataFrame(
    [
        ("India", 0.18),
        ("Sri Lanka", 0.15),
        ("Nepal", 0.13),
        ("Bhutan", 0.10),
    ],
    ["country", "tax_rate"],
)

skewed_join = (
    order_fact
    .join(country_rules, "country")
    .select(
        F.col("order_id"),
        F.col("country"),
        (
            F.col("order_amount") * (1 + F.col("tax_rate"))
        ).alias("gross_amount"),
    )
)

skewed_join.explain(mode="formatted")

(
    skewed_join
    .agg(F.sum("gross_amount").alias("total_gross_amount"))
    .show()
)

# 12. Prefer broadcast when one side is tiny

Broadcast sends the four-row lookup to every executor and streams existing fact partitions locally. It removes the country shuffle and the India reducer. This is simpler than salting. Broadcast only when the fully materialized build side safely fits executor memory; a hint overrides the threshold decision.

In [ ]:
broadcast_join = (
    order_fact
    .join(F.broadcast(country_rules), "country")
    .select(
        F.col("order_id"),
        F.col("country"),
        (
            F.col("order_amount") * (1 + F.col("tax_rate"))
        ).alias("gross_amount"),
    )
)

broadcast_join.explain(mode="formatted")

(
    broadcast_join
    .agg(F.sum("gross_amount").alias("total_gross_amount"))
    .show()
)

# 13. Salt a shuffle join when broadcast is impossible

Salt the large side and replicate each lookup row across every salt. Join on `(country, salt)`. The India key is now spread across reducers. Replication expands the other side by `SALT_BUCKETS`, so this is dangerous if both sides are large. Outer joins require extra handling for unmatched-row semantics.

In [ ]:
large_salted = (
    order_fact
    .withColumn(
        "salt",
        F.pmod(F.xxhash64("order_id"), F.lit(SALT_BUCKETS)),
    )
)

rules_salted = (
    country_rules
    .withColumn(
        "salt",
        F.explode(
            F.sequence(F.lit(0), F.lit(SALT_BUCKETS - 1))
        ),
    )
)

salted_join = (
    large_salted
    .join(rules_salted, ["country", "salt"])
    .select(
        F.col("order_id"),
        F.col("country"),
        (
            F.col("order_amount") * (1 + F.col("tax_rate"))
        ).alias("gross_amount"),
    )
)

salted_join.explain(mode="formatted")

(
    salted_join
    .agg(F.sum("gross_amount").alias("total_gross_amount"))
    .show()
)

# 14. Enable AQE and configure skew detection

At shuffle query-stage boundaries AQE receives actual map-output sizes. For a sort-merge join, a partition is skewed only when it is both larger than `median partition size × skewedPartitionFactor` and larger than `skewedPartitionThresholdInBytes`. Spark can split the large-side partition and replicate matching data from the other side.

The low thresholds below make the behavior visible in a teaching workload. Production thresholds should follow measured partition sizes; `forceOptimizeSkewedJoin` may introduce an extra shuffle and must be benchmarked.

In [ ]:
aqe_settings = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor": "2.0",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes": "8MB",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes": "4MB",
    "spark.sql.adaptive.forceOptimizeSkewedJoin": "true",
    "spark.sql.autoBroadcastJoinThreshold": "-1",
    "spark.sql.adaptive.autoBroadcastJoinThreshold": "-1",
}

for setting_name, setting_value in aqe_settings.items():
    spark.conf.set(setting_name, setting_value)

print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))

# 15. AQE before and after execution

Build a new DataFrame after changing configuration. Before the action expect `AdaptiveSparkPlan isFinalPlan=false`. After execution, inspect the final plan and SQL UI for adaptive/skew join readers and tasks sharing the former hot partition. The country lookup is intentionally forced to shuffle only to reveal AQE; broadcast remains its normal production solution.

In [ ]:
aqe_join = (
    order_fact
    .join(country_rules, "country")
    .select(
        F.col("country"),
        (
            F.col("order_amount") * (1 + F.col("tax_rate"))
        ).alias("gross_amount"),
    )
    .groupBy("country")
    .agg(F.sum("gross_amount").alias("gross_revenue"))
)

print("BEFORE ACTION")
aqe_join.explain(mode="formatted")

started = perf_counter()
result = aqe_join.collect()
elapsed_seconds = perf_counter() - started

print(f"Elapsed: {elapsed_seconds:.3f} seconds")
print("Result:", result)
print("AFTER ACTION")
aqe_join.explain(mode="formatted")

# 16. Hot-customer AQE experiment

Twenty percent of orders use customer 0. This creates a genuine hot `customer_id` shuffle key. It demonstrates that geographic skew and customer-key skew are different: 99% India does not by itself skew hashing on well-distributed customer IDs, while a default/unknown/test customer ID can. Null and default keys are common real-world causes.

In [ ]:
(
    orders
    .groupBy("customer_id")
    .count()
    .orderBy(F.desc("count"))
    .show(10)
)

aqe_customer_join = (
    orders
    .join(
        customers.select("customer_id", "country"),
        "customer_id",
    )
    .groupBy("country")
    .agg(F.count("*").alias("orders"))
)

print("BEFORE ACTION")
aqe_customer_join.explain(mode="formatted")

customer_join_result = aqe_customer_join.collect()
print("Result:", customer_join_result)

print("AFTER ACTION")
aqe_customer_join.explain(mode="formatted")

# 17. What AQE can and cannot fix

| Situation | Result |
|---|---|
| Skewed sort-merge join partition | AQE can split it and replicate the matching side |
| Small post-shuffle partitions | AQE can coalesce them |
| Join side becomes small at runtime | AQE can change some join strategies |
| One hot final aggregation key | Skew-join handling does not generally split it; use two-stage aggregation |
| Expensive UDF with balanced bytes | Improve/profile computation; AQE byte thresholds may not see it |
| Skewed historical files | Rewrite layout or compact; AQE does not maintain storage |
| Incorrect many-to-many join | Fix the data model; AQE cannot fix semantics |

AQE is reactive and begins only after query-stage statistics exist. It complements—not replaces—correct keys, filtering, projection, good file layout, and measurement.

# 18. Benchmark and correctness checklist

1. Use identical staged inputs and identical terminal actions.
2. Label runs: AQE off, broadcast, salt, country-state hierarchy, AQE on.
3. Compare median and maximum task time, shuffle bytes, spill, memory, and output rows.
4. Account for warm filesystem caches and JVM/code-generation warm-up.
5. Verify row counts and revenue totals after every rewrite.
6. Prefer the simplest correct solution that improves repeated measurements.

A salted plan can be slower when Spark's native partial aggregation already neutralizes skew. Optimization follows evidence.

## Optional cleanup

Staged files remain for repeatable comparisons. The cleanup is commented and protected by an exact-path assertion.

In [ ]:
# Uncomment only after completing all experiments.
# assert STAGE_ROOT == "hdfs://localhost:9000/tmp/dataeng/skew_lab"
#
# filesystem = (
#     spark._jvm.org.apache.hadoop.fs.FileSystem
#     .get(spark._jsc.hadoopConfiguration())
# )
# stage_path = spark._jvm.org.apache.hadoop.fs.Path(STAGE_ROOT)
#
# if filesystem.exists(stage_path):
#     filesystem.delete(stage_path, True)
#
# spark.stop()